In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics import cohen_kappa_score
from scipy import stats

# ── ПУТИ ──────────────────────────────────────────────────────────
ZS_PATH   = '/content/drive/MyDrive/SFU 4/VKR/3. labeling/Разметка (zero shot)_2/zeroshot_300_results.csv'
DICT_PATH = '/content/drive/MyDrive/SFU 4/VKR/3. labeling/Словарная разметка/(FOR USE) После удаления одинаковых/corpus_labeled_full_cleaned.csv'
# ──────────────────────────────────────────────────────────────────

df_zs   = pd.read_csv(ZS_PATH)
df_dict = pd.read_csv(DICT_PATH)

df_dict['filename_clean'] = df_dict['filename'].str.replace('lemma_', '', regex=False)
df_zs['filename_clean']   = df_zs['filename']

merged = df_zs.merge(
    df_dict[['filename_clean', 'label']],
    on='filename_clean', how='inner'
).rename(columns={'label': 'label_dict'})

print(f'Текстов для анализа: {len(merged)}')

# ── Cohen's Kappa ──────────────────────────────────────────────────
kappa = cohen_kappa_score(merged['label_dict'], merged['label_zeroshot'])
n = len(merged)

# ── p-value через z-тест ──────────────────────────────────────────
# Стандартная ошибка Kappa
labels = ['depression', 'anxiety', 'neutral']

# Матрица наблюдаемых частот
observed = pd.crosstab(merged['label_dict'], merged['label_zeroshot'])
observed = observed.reindex(index=labels, columns=labels, fill_value=0)

# Маргинальные вероятности
p_row = observed.sum(axis=1) / n  # словарный
p_col = observed.sum(axis=0) / n  # zero-shot

# Ожидаемое согласование по случайности
p_e = (p_row * p_col).sum()
p_o = (merged['label_dict'] == merged['label_zeroshot']).mean()

# Стандартная ошибка (формула Fleiss et al.)
se_kappa = np.sqrt((p_o * (1 - p_o)) / (n * (1 - p_e)**2))

# z-статистика и p-value
z = kappa / se_kappa
p_value = 2 * (1 - stats.norm.cdf(abs(z)))

print('\n══════════════════════════════════════════')
print(f'  Согласованность (P_o):  {p_o*100:.1f}%')
print(f'  Случайное согласование: {p_e*100:.1f}%')
print(f'  Cohen\'s Kappa:          {kappa:.4f}')
print(f'  Стандартная ошибка:     {se_kappa:.4f}')
print(f'  z-статистика:           {z:.4f}')
print(f'  p-value:                {p_value:.6f}')
print(f'  Значимо (p < 0.05):     {"ДА ✓" if p_value < 0.05 else "НЕТ ✗"}')
print('══════════════════════════════════════════')

if p_value < 0.001:
    print('  Интерпретация: согласованность высоко значима (p < 0.001)')
elif p_value < 0.05:
    print('  Интерпретация: согласованность значима (p < 0.05)')
else:
    print('  Интерпретация: согласованность не отличается от случайной')

# ── Дополнительно: хи-квадрат тест ───────────────────────────────
# Проверяем независимость разметок двух методов
from scipy.stats import chi2_contingency
chi2, p_chi2, dof, expected = chi2_contingency(observed)
print(f'\n── Хи-квадрат тест ──')
print(f'  χ² = {chi2:.4f}')
print(f'  df = {dof}')
print(f'  p-value = {p_chi2:.6f}')
print(f'  Значимо (p < 0.05): {"ДА ✓" if p_chi2 < 0.05 else "НЕТ ✗"}')

Текстов для анализа: 276

══════════════════════════════════════════
  Согласованность (P_o):  40.6%
  Случайное согласование: 28.4%
  Cohen's Kappa:          0.1697
  Стандартная ошибка:     0.0413
  z-статистика:           4.1091
  p-value:                0.000040
  Значимо (p < 0.05):     ДА ✓
══════════════════════════════════════════
  Интерпретация: согласованность высоко значима (p < 0.001)

── Хи-квадрат тест ──
  χ² = 39.4165
  df = 4
  p-value = 0.000000
  Значимо (p < 0.05): ДА ✓
